## Building Multi Step Workflows

This looks like a great lesson on orchestrating serverless microservices! I’ve cleaned up the formatting using proper Markdown headers, code blocks, and visual cues to make the flow easier for students to follow.

---

# Introduction: The Need for Multi-Step Workflows

Welcome back! In the last lesson, you learned how to build and deploy simple serverless APIs using Google Cloud Functions and API Gateway. You saw how a single Cloud Function can handle a request and return a response.

However, many real-world applications require more than just a single step. For example, when processing an online order, you might need to:

1.  **Validate** the order details.
2.  **Charge** the customer.
3.  **Record** the order in a database.



Each of these steps could be handled by a separate Cloud Function. But how do you make sure they run in the right order and that data flows smoothly from one step to the next? This is where **Google Cloud Workflows** come in. Workflows let you organize and manage multi-step processes in your serverless applications.

---

## Quick Recall: Cloud Functions in Serverless Apps

Before we dive into Workflows, let's quickly remind ourselves how Cloud Functions fit into serverless applications. A **Cloud Function** is a small piece of code that runs in response to an event, such as an API call or a message. 

In the previous lesson, you learned how to:
* Write a Cloud Function that processes input and returns output.
* Connect a Cloud Function to an API Gateway endpoint.

In this lesson, we will see how to connect several Cloud Functions together to handle a more complex process.

---

## Understanding Google Cloud Workflows

Google Cloud Workflows help you coordinate multiple Cloud Functions (or other Google Cloud services) into a single workflow. A workflow is like a **flowchart**: it defines a series of steps and the order in which they run. Each step can perform a task, make a choice, or wait for something to happen.

### The Order Processing Example
Imagine you have three steps:
1.  **Validate the order** (check if the amount is valid).
2.  **Charge the customer** (simulate payment).
3.  **Record the order** (save it to a database).

With Workflows, you can define a process that runs these steps one after another, passing the result from one step to the next.

---

## Breaking Down the Order Workflow Example

Let's build up the order processing workflow step-by-step using three Cloud Functions.

### Step 1: Validate the Order
First, we need a Cloud Function to check if the order amount is valid.

```python
# main.py (validate_order function)
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    if float(order["amount"]) <= 0:
        return {"error": "Invalid amount"}, 400
    order["validated"] = True
    return {"order": order}
```

* **Logic:** If the amount is valid, it adds a `"validated": True` field.
* **Input Example:** `{"order": {"order_id": "test-123", "amount": "99.99"}}`
* **Output Example:** `{"order": {"order_id": "test-123", "amount": "99.99", "validated": true}}`

### Step 2: Charge the Customer
Next, we simulate charging the customer using the output from Step 1.

```python
# main.py (charge_customer function)
def charge_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    request_json["payment_id"] = f"pay_{order['order_id']}"
    request_json["charged"] = True
    return request_json
```

* **Output Example:** `{"order": {...}, "payment_id": "pay_test-123", "charged": true}`

### Step 3: Record the Order
Finally, we save the order to a **Firestore** collection.

```python
# main.py (record_order function)
import time
from google.cloud import firestore

def record_order(request):
    db = firestore.Client()
    request_json = request.get_json()
    order = request_json["order"]
    item = {
        "order_id": order["order_id"],
        "amount": float(order["amount"]),
        "payment_id": request_json["payment_id"],
        "ts": int(time.time())
    }
    db.collection("orders").document(order["order_id"]).set(item)
    return {"status": "ok", "order": item}
```

---

## Defining the Workflow (`YAML`)

This `order_workflow.yaml` file acts as the "glue" that connects our functions.

```yaml
main:
  params: [input]
  steps:
    - validate:
        call: http.post
        args:
          url: https://REGION-PROJECT_ID.cloudfunctions.net/validate_order
          body: ${input}
        result: validated
    - charge:
        call: http.post
        args:
          url: https://REGION-PROJECT_ID.cloudfunctions.net/charge_customer
          body: ${validated.body}
        result: charged
    - record:
        call: http.post
        args:
          url: https://REGION-PROJECT_ID.cloudfunctions.net/record_order
          body: ${charged.body}
        result: recorded
    - return_output:
        return: ${recorded.body}
```

---

## Deploying and Testing

### 1. Deploy Cloud Functions
```bash
gcloud functions deploy validate_order --runtime python310 --trigger-http --allow-unauthenticated
gcloud functions deploy charge_customer --runtime python310 --trigger-http --allow-unauthenticated
gcloud functions deploy record_order --runtime python310 --trigger-http --allow-unauthenticated
```

### 2. Deploy the Workflow
```bash
gcloud workflows deploy order-workflow --source=order_workflow.yaml --location=REGION
```

### 3. Execute the Workflow
Trigger the workflow with sample data:
```bash
gcloud workflows execute order-workflow \
  --data='{"order": {"order_id": "test-123", "amount": "99.99"}}' \
  --location=REGION
```

**Expected Final Output:**
```json
{
  "status": "ok", 
  "order": {
    "order_id": "test-123", 
    "amount": 99.99, 
    "payment_id": "pay_test-123", 
    "ts": 1710000000
  }
}
```

---

## Summary and Practice Preview

In this lesson, you learned how to:
* **Deconstruct** a complex process into modular Cloud Functions.
* **Orchestrate** those modules using a YAML-based workflow.
* **Manage State** by passing variables from one step to the next.

Next, you'll get hands-on practice building your own Workflows! Good luck!

## Fixing Order Amount Validation in Cloud Function

You have a multi-step order processing workflow using Google Cloud Workflows and Cloud Functions. There is a bug in the validation step: the system is currently accepting orders with zero amounts as valid, which should not happen.

Your task is to examine the validate_order function and fix the conditional statement so that orders with exactly zero amounts are properly rejected.

Test your fix with these cases:

    Orders with negative amounts should be rejected (already working)
    Orders with zero amounts should be rejected (currently broken)
    Orders with positive amounts should be accepted (already working)

Debugging and fixing validation logic is essential for building reliable serverless workflows.

```python
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    if float(order["amount"]) < 0:
        return {"error": "Invalid amount"}, 400
    order["validated"] = True
    return {"order": order}
    
def charge_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    request_json["payment_id"] = f"pay_{order['order_id']}"
    request_json["charged"] = True
    return request_json
    
import time
from google.cloud import firestore

def record_order(request):
    db = firestore.Client()
    request_json = request.get_json()
    order = request_json["order"]
    item = {
        "order_id": order["order_id"],
        "amount": float(order["amount"]),
        "payment_id": request_json["payment_id"],
        "ts": int(time.time())
    }
    db.collection("orders").document(order["order_id"]).set(item)
    return {"status": "ok", "order": item}

```

The issue in your `validate_order` function lies in the comparison operator. Currently, it only checks if the amount is strictly less than zero ($< 0$), which allows $0$ to slip through as a valid value.

To fix this, you should change the condition to check if the amount is **less than or equal to zero** ($\le 0$).

### Fixed Code

```python
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    
    # FIX: Changed < to <= to reject zero-amount orders
    if float(order["amount"]) <= 0:
        return {"error": "Invalid amount"}, 400
        
    order["validated"] = True
    return {"order": order}
```

---

### Why this fix works:

* **Negative Amounts (-5.00):** These are still caught because $-5 \le 0$ is **True**.
* **Zero Amounts (0.00):** These are now caught because $0 \le 0$ is **True**.
* **Positive Amounts (10.00):** These are still accepted because $10 \le 0$ is **False**, allowing the function to proceed to set `order["validated"] = True`.

### Pro-Tip for Production
When dealing with financial data, it's also a good idea to wrap your `float()` conversion in a `try/except` block. If the input is missing or is a non-numeric string (like `"abc"`), the function will crash with a `500 Internal Server Error` instead of returning a clean `400 Bad Request` error message to your workflow.

## Preserving Data Fields Across Workflow Steps in Cloud Functions

Now that you understand how Google Cloud Workflows coordinate multiple Cloud Functions, it's time to practice maintaining data flow across your workflow steps. Currently, our workflow handles basic order information, but we need to add support for tracking customer_email addresses throughout the entire process.

Your objective is to modify all three Cloud Functions to handle a new customer_email field that will be included in the original order input. Here's what you need to do:

    Update the validate_order function to preserve the customer_email field in the order object
    Update the charge_customer function to keep the customer_email field in the request data
    Update the record_order function to store the customer_email in the Firestore document

The key concept here is understanding how data moves from one step to the next in a workflow. Each function receives the output of the previous function as its input, so you need to make sure important data isn't lost along the way.

Test your workflow with input like this:

JSON
{"order": {"order_id": "test-123", "amount": "99.99", "customer_email": "customer@example.com"}}

When everything works correctly, you should see the customer email stored in your Firestore collection alongside the order details. This exercise will help you master data continuity in serverless workflows — a skill you'll use in every real-world Cloud Workflows application.

```python
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    if float(order["amount"]) <= 0:
        return {"error": "Invalid amount"}, 400
    order["validated"] = True
    # TODO: Make sure the customer_email field is preserved in the order object
    return {"order": order}

def charge_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    request_json["payment_id"] = f"pay_{order['order_id']}"
    request_json["charged"] = True
    # TODO: Ensure all data including customer_email flows through to the next step
    return request_json

import time
from google.cloud import firestore

def record_order(request):
    db = firestore.Client()
    request_json = request.get_json()
    order = request_json["order"]
    # TODO: Add the customer_email field to the Firestore document
    item = {
        "order_id": order["order_id"],
        "amount": float(order["amount"]),
        "payment_id": request_json["payment_id"],
        "ts": int(time.time())
    }
    db.collection("orders").document(order["order_id"]).set(item)
    return {"status": "ok", "order": item}
```

## Adding Error Handling to a Google Cloud Workflow

Your order processing workflow currently has no error handling. When a Cloud Function fails (like validation rejecting an invalid order), the workflow crashes instead of handling the error gracefully.

Your task is to add try/except blocks to the validate workflow step in order_workflow.yaml so that errors are caught and returned instead of crashing the workflow. This allows you to see what went wrong and handle failures appropriately.

For the validate workflow step:

    Wrap the existing http.post call in a try block
    Add an except block that catches errors as e and returns them in a return_error step

Test your implementation with both valid and invalid orders. Try an order with a negative or zero amount to trigger a validation error and see how your error handling works.

Building robust error handling is critical for production serverless workflows.


```yaml
main:
  params: [input]
  steps:
    - validate:
        call: http.post
        args:
          url: http://localhost:3001
          body: ${input}
        result: validated
        # TODO: Wrap this call in a try/except block to handle errors
    - charge:
        try:
          call: http.post
          args:
            url: http://localhost:3002
            body: ${validated.body}
          result: charged
        except:
          as: e
          steps:
            - return_error:
                return: ${e}
    - record:
        try:
          call: http.post
          args:
            url: http://localhost:3003
            body: ${charged.body}
          result: recorded
        except:
          as: e
          steps:
            - return_error:
                return: ${e}
    - return_output:
        return: ${recorded.body}
```

Adding `try/except` blocks to your workflow steps is like installing a safety net. Without them, if a Cloud Function returns a status code like **400** (Bad Request) or **500** (Server Error), the entire workflow execution terminates immediately with a "Failed" status.

By catching these errors, you can transform a system crash into a controlled error message.

---

### Updated `order_workflow.yaml`

Here is the corrected YAML structure. Note that in Google Cloud Workflows, the `try` block wraps the logic, and the `except` block lives at the same indentation level as the `try`.

```yaml
main:
  params: [input]
  steps:
    - validate:
        try:
          call: http.post
          args:
            url: https://REGION-PROJECT_ID.cloudfunctions.net/validate_order
            body: ${input}
          result: validated
        except:
          as: e
          steps:
            - return_validation_error:
                # This returns the error details (body, code, etc.) instead of crashing
                return: ${e}
    - charge:
        try:
          call: http.post
          args:
            url: https://REGION-PROJECT_ID.cloudfunctions.net/charge_customer
            body: ${validated.body}
          result: charged
        except:
          as: e
          steps:
            - return_charge_error:
                return: ${e}
    - record:
        try:
          call: http.post
          args:
            url: https://REGION-PROJECT_ID.cloudfunctions.net/record_order
            body: ${charged.body}
          result: recorded
        except:
          as: e
          steps:
            - return_record_error:
                return: ${e}
    - return_output:
        return: ${recorded.body}
```

---

### What happens when an error occurs?



1.  **The Trigger:** If you send `{"order": {"amount": 0}}`, the `validate_order` function returns a **400 Bad Request**.
2.  **The Exception:** The workflow engine sees the 400 code and immediately jumps to the `except` block.
3.  **The Variable:** The variable `e` (defined by `as: e`) captures a dictionary containing:
    * `e.code`: The HTTP status code (e.g., 400).
    * `e.body`: The actual error message from your Python code (e.g., `{"error": "Invalid amount"}`).
4.  **The Result:** The workflow finishes with a "Succeeded" state (because you handled the error), but the output provides the error details so the user knows exactly what went wrong.

### Why this is essential
In production, you might not just "return" the error. You could use the `except` block to:
* **Retry** the function (if it was a temporary network glitch).
* **Log** the failure to Cloud Logging.
* **Notify** a developer via an email or Slack alert.

## Adding a Notification Step to a Google Cloud Workflow

You have a multi-step order processing workflow using Google Cloud Workflows and Cloud Functions. The workflow currently validates orders, charges customers, and records orders in Firestore.

Your task is to extend this workflow by adding a new notification step that sends a confirmation message to the customer after their order has been successfully recorded.

You need to:

    Implement the notify_customer function in main.py that extracts order details and creates a confirmation message
    Update open_order_workflow.yaml to include the new notification step after the record step
    The notification should include the order_id, amount, and payment_id
    The workflow should return the notification result as the final output

Test your implementation with the provided event in event.json. The workflow should process the order through all four steps and return a notification confirmation.

Extending multi-step workflows is a common task in serverless application development.

```python
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    if float(order["amount"]) <= 0:
        return {"error": "Invalid amount"}, 400
    order["validated"] = True
    return {"order": order}

def charge_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    request_json["payment_id"] = f"pay_{order['order_id']}"
    request_json["charged"] = True
    return request_json
    
import time
from google.cloud import firestore

def record_order(request):
    db = firestore.Client()
    request_json = request.get_json()
    order = request_json["order"]
    item = {
        "order_id": order["order_id"],
        "amount": float(order["amount"]),
        "payment_id": request_json["payment_id"],
        "ts": int(time.time())
    }
    db.collection("orders").document(order["order_id"]).set(item)
    return {"status": "ok", "order": item}

# TODO: Implement the notify_customer function
def notify_customer(request):
    # TODO: Extract the request JSON
    # TODO: Extract the order from the request
    
    # TODO: Create a confirmation message including order_id, amount, and payment_id
    # TODO: Print the confirmation message
    
    # TODO: Return a response with status "notification_sent" and the message
    pass


```

```yaml
main:
  params: [input]
  steps:
    - validate:
        try:
          call: http.post
          args:
            url: http://localhost:3001
            body: ${input}
          result: validated
        except:
          as: e
          steps:
            - return_validation_error:
                return: ${e}
    - charge:
        try:
          call: http.post
          args:
            url: http://localhost:3002
            body: ${validated.body}
          result: charged
        except:
          as: e
          steps:
            - return_charge_error:
                return: ${e}
    - record:
        try:
          call: http.post
          args:
            url: http://localhost:3003
            body: ${charged.body}
          result: recorded
        except:
          as: e
          steps:
            - return_record_error:
                return: ${e}
    - notify:
        try:
          call: http.post
          args:
            url: http://localhost:3004
            body: ${recorded.body}
          result: notified
        except:
          as: e
          steps:
            - return_notify_error:
                return: ${e}
    - return_output:
        return: ${notified.body}

```

```json
{"order": {"order_id": "test-123", "amount": "99.99"}}
```

## Building a Multi-Step Returns Workflow

You have a multi-step product returns processing workflow using Google Cloud Workflows and Cloud Functions. There is a bug in the return policy validation step: the system is currently rejecting returns made on exactly the 30th day after purchase, when these should actually be accepted according to the 30-day return policy.

Your task is to examine the check_return_policy function and fix the conditional statement so that returns made on day 30 are properly accepted.

Test your fix with these cases:

    Returns beyond 30 days should be rejected (already working)
    Returns on exactly day 30 should be accepted (currently broken - they're being rejected)
    Returns within 30 days should be accepted (already working)

Debugging and fixing validation logic is essential for building reliable serverless workflows.

```python
def check_return_policy(request):
    request_json = request.get_json()
    return_data = request_json["return"]
    
    if int(return_data["days_since_purchase"]) >= 30:
        return {"error": "Return period expired"}, 400
    
    valid_reasons = ["defective", "wrong_item", "not_as_described"]
    if return_data["reason"] not in valid_reasons:
        return {"error": "Invalid return reason"}, 400
    
    return_data["policy_approved"] = True
    return {"return": return_data}
    
def process_refund(request):
    request_json = request.get_json()
    return_data = request_json["return"]
    request_json["refund_id"] = f"ref_{return_data['return_id']}"
    request_json["refund_amount"] = return_data["amount"]
    request_json["refund_processed"] = True
    return request_json
    
import time
from google.cloud import firestore

def update_inventory(request):
    db = firestore.Client()
    request_json = request.get_json()
    return_data = request_json["return"]
    item = {
        "return_id": return_data["return_id"],
        "order_id": return_data["order_id"],
        "refund_id": request_json["refund_id"],
        "refund_amount": float(request_json["refund_amount"]),
        "reason": return_data["reason"],
        "ts": int(time.time())
    }
    db.collection("returns").document(return_data["return_id"]).set(item)
    return {"status": "inventory_updated", "return": item}

```

```yaml
main:
  params: [input]
  steps:
    - check_policy:
        try:
          call: http.post
          args:
            url: http://localhost:3001
            body: ${input}
          result: validated
        except:
          as: e
          steps:
            - return_error:
                return: ${e}
    - refund:
        try:
          call: http.post
          args:
            url: http://localhost:3002
            body: ${validated.body}
          result: refunded
        except:
          as: e
          steps:
            - return_error:
                return: ${e}
    - inventory:
        try:
          call: http.post
          args:
            url: http://localhost:3003
            body: ${refunded.body}
          result: recorded
        except:
          as: e
          steps:
            - return_error:
                return: ${e}
    - return_output:
        return: ${recorded.body}
```

```json
{"return": {"return_id": "ret-123", "order_id": "ord-456", "days_since_purchase": 30, "reason": "defective", "amount": 99.99}}
```